In [ ]:
# ============================================================
# DIABETES READMISSION PREDICTION MODEL
# Full End-to-End Machine Learning Pipeline
# ============================================================

import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

# =========================
# LOAD DATASET
# =========================

DATASET_PATH = "diabetic_data.csv"

print("Loading dataset...")

# Load CSV File

df = pd.read_csv(DATASET_PATH)

print("Dataset Loaded Successfully")
print("Dataset Shape:", df.shape)

# =========================
# DISPLAY BASIC INFORMATION
# =========================

print("\nFirst 5 Rows:")
print(df.head())

print("\nDataset Information:")
print(df.info())

print("\nMissing Values:")
print(df.isnull().sum())

# =========================
# HANDLE MISSING VALUES
# =========================

# Replace '?' with NaN

df.replace('?', np.nan, inplace=True)

# =========================
# REMOVE UNNECESSARY COLUMNS
# =========================

columns_to_drop = [
    'encounter_id',
    'patient_nbr'
]

for col in columns_to_drop:
    if col in df.columns:
        df.drop(col, axis=1, inplace=True)

print("\nColumns after dropping unnecessary columns:")
print(df.columns)

# =========================
# TARGET COLUMN
# =========================

TARGET_COLUMN = 'readmitted'

print("\nTarget Distribution:")
print(df[TARGET_COLUMN].value_counts())

# =========================
# CONVERT TARGET VALUES
# =========================

# NO -> 0
# <30 and >30 -> 1

df[TARGET_COLUMN] = df[TARGET_COLUMN].apply(
    lambda x: 0 if x == 'NO' else 1
)

print("\nConverted Target Distribution:")
print(df[TARGET_COLUMN].value_counts())

# =========================
# TARGET DISTRIBUTION GRAPH
# =========================

plt.figure(figsize=(6, 4))

df[TARGET_COLUMN].value_counts().plot(kind='bar')

plt.title('Target Variable Distribution')
plt.xlabel('Readmission')
plt.ylabel('Count')

plt.tight_layout()
plt.show()

# =========================
# SPLIT FEATURES AND TARGET
# =========================

X = df.drop(TARGET_COLUMN, axis=1)
y = df[TARGET_COLUMN]

print("\nFeature Shape:", X.shape)
print("Target Shape:", y.shape)

# =========================
# IDENTIFY FEATURE TYPES
# =========================

numeric_features = X.select_dtypes(
    include=['int64', 'float64']
).columns.tolist()

categorical_features = X.select_dtypes(
    include=['object']
).columns.tolist()

print("\nNumber of Numeric Features:", len(numeric_features))
print("Number of Categorical Features:", len(categorical_features))

# =========================
# PREPROCESSING PIPELINES
# =========================

# Numeric Pipeline

numeric_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]
)

# Categorical Pipeline

categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]
)

# Combined Preprocessor

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# =========================
# TRAIN TEST SPLIT
# =========================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining Set Shape:", X_train.shape)
print("Testing Set Shape:", X_test.shape)

# =========================
# CREATE MACHINE LEARNING MODEL
# =========================

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

# =========================
# CREATE FULL PIPELINE
# =========================

pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ]
)

# =========================
# TRAIN MODEL
# =========================

print("\nTraining Model...")

pipeline.fit(X_train, y_train)

print("Model Training Completed")

# =========================
# MAKE PREDICTIONS
# =========================

print("\nMaking Predictions...")

y_pred = pipeline.predict(X_test)

# =========================
# MODEL EVALUATION
# =========================

accuracy = accuracy_score(y_test, y_pred)

print("\n==============================")
print("MODEL EVALUATION")
print("==============================")

print(f"Accuracy: {accuracy * 100:.2f}%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# =========================
# CONFUSION MATRIX
# =========================

cm = confusion_matrix(y_test, y_pred)

print("\nConfusion Matrix:")
print(cm)

plt.figure(figsize=(6, 6))

ConfusionMatrixDisplay(
    confusion_matrix=cm
).plot()

plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

# =========================
# ROC CURVE
# =========================

plt.figure(figsize=(7, 6))

RocCurveDisplay.from_estimator(
    pipeline,
    X_test,
    y_test
)

plt.title('ROC Curve')
plt.tight_layout()
plt.show()

# =========================
# SAVE MODEL
# =========================

MODEL_FILE_NAME = 'diabetic_model.pkl'

with open(MODEL_FILE_NAME, 'wb') as file:
    pickle.dump(pipeline, file)

print(f"\nModel saved successfully as: {MODEL_FILE_NAME}")

# =========================
# LOAD SAVED MODEL
# =========================

with open(MODEL_FILE_NAME, 'rb') as file:
    loaded_model = pickle.load(file)

print("Model loaded successfully")

# =========================
# SAMPLE PREDICTION
# =========================

sample_data = X.iloc[[0]]

prediction = loaded_model.predict(sample_data)
probability = loaded_model.predict_proba(sample_data)

print("\n==============================")
print("SAMPLE PREDICTION")
print("==============================")

print("Prediction:", prediction[0])
print("Probability:", probability)

if prediction[0] == 1:
    print("Patient is likely to be readmitted")
else:
    print("Patient is not likely to be readmitted")

# =========================
# FEATURE IMPORTANCE
# =========================

print("\n==============================")
print("FEATURE IMPORTANCE")
print("==============================")

classifier = pipeline.named_steps['classifier']

encoded_features = pipeline.named_steps[
    'preprocessor'
].get_feature_names_out()

feature_importance = pd.DataFrame({
    'Feature': encoded_features,
    'Importance': classifier.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by='Importance',
    ascending=False
)

print(feature_importance.head(20))

# =========================
# FEATURE IMPORTANCE GRAPH
# =========================

top_features = feature_importance.head(10)

plt.figure(figsize=(12, 6))

plt.barh(
    top_features['Feature'],
    top_features['Importance']
)

plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.title('Top 10 Important Features')

plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

# =========================
# SAVE FEATURE IMPORTANCE
# =========================

feature_importance.to_csv(
    'feature_importance.csv',
    index=False
)

print("\nFeature importance saved successfully")

# =========================
# END OF PROJECT
# =========================

print("\nProject Completed Successfully")